# 05 — BERT (BERTimbau, FinBERT-PT-BR, DeB3RTa) com RandomizedSearchCV

Loop sobre os 3 modelos da `MODEL_REGISTRY`. Por modelo: busca de hiperparametros (binario + multiclasse) em train+val (val unico, sem inner-CV) seguida pelos 6 regimes com os melhores hiperparametros.

**Assimetria declarada**: a busca BERT usa val unico para selecao (custo computacional do inner-CV proibitivo). A busca TF-IDF (notebooks 02-04) usa StratifiedKFold(5) interno. Esta diferenca esta registrada em cada `result_card.json` no campo `hyperparameter_search`.

**Hardware**: A100 recomendado. Em T4 multiplique tempos por 3-5x ou reduza `N_ITER_BERT`/`MODELS`.

## 0. Verificacao de GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU nao detectada. Va em Runtime > Change runtime type > GPU (A100 recomendado).",
    )

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
print(f"GPU: {GPU_NAME} ({VRAM_GB} GB VRAM)")
print(f"CUDA: {torch.version.cuda}")

# A100 recomendado para o orcamento de busca abaixo. Em T4 multiplique tempos por ~3-5x.
HARDWARE = f"Colab-{GPU_NAME.split()[-1]}"
if "A100" not in GPU_NAME:
    print(f"\nAVISO: {GPU_NAME} detectada. Os defaults (N_ITER_BERT=25 trials x 3 models x 2 tasks) "
          f"foram dimensionados para A100. Em GPUs menores reduza N_ITER_BERT ou MODELS.")


## 1. Bootstrap (Colab + local)

In [ ]:
import subprocess
import sys
import zipfile
from pathlib import Path


def _run(cmd: list[str], description: str) -> None:
    """Run a subprocess command, print stdout/stderr, raise on non-zero exit."""
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr, file=sys.stderr)
        raise RuntimeError(f"{description} failed with exit code {result.returncode}")


IN_COLAB = "google.colab" in sys.modules
print("Ambiente:", "Google Colab" if IN_COLAB else "Local")
print("Python   :", sys.version.split()[0])

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/almeidadm/economy-classifier.git"
    REPO_BRANCH = "main"
    DRIVE_FOLDER = "economy-classifier"

    DRIVE_BASE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    REPO_DIR = Path("/content/economy-classifier")

    if REPO_DIR.exists():
        # Pull latest so a re-run picks up commits without manual rm -rf
        _run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], "git fetch")
        _run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], "git checkout")
        _run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], "git reset")
    else:
        _run(
            ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
            "git clone",
        )

    # Install only what's missing — preserve Colab's preinstalled torch/transformers
    _run(
        [sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR),
         "--upgrade-strategy", "only-if-needed", "-q"],
        "pip install -e .",
    )

    # Make the editable install importable in the current kernel session
    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))

    SPLITS_DIR = REPO_DIR / "artifacts" / "splits"
    if not (SPLITS_DIR / "train.parquet").exists():
        zip_path = DRIVE_BASE / "colab_splits.zip"
        assert zip_path.exists(), f"Falta colab_splits.zip em {DRIVE_BASE}."
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(REPO_DIR / "artifacts")

    RUNS_BASE = DRIVE_BASE / "runs"
    SEARCH_TRIALS_BASE = Path("/content/bert_search_trials")  # ephemeral on Colab
else:
    REPO_DIR = Path.cwd().parent
    SPLITS_DIR = REPO_DIR / "artifacts" / "splits"
    RUNS_BASE = REPO_DIR / "artifacts" / "runs"
    SEARCH_TRIALS_BASE = REPO_DIR / "artifacts" / "_bert_search_trials"

RUNS_BASE.mkdir(parents=True, exist_ok=True)
SEARCH_TRIALS_BASE.mkdir(parents=True, exist_ok=True)
print("SPLITS_DIR        :", SPLITS_DIR)
print("RUNS_BASE         :", RUNS_BASE)
print("SEARCH_TRIALS_BASE:", SEARCH_TRIALS_BASE)
print("HARDWARE          :", HARDWARE)


def _sanity_check_imports() -> None:
    """Fail loud at bootstrap if scientific stack is inconsistent (common Colab hazard)."""
    failures = []
    for mod_name in ("numpy", "scipy", "pandas", "sklearn", "torch", "transformers", "economy_classifier"):
        try:
            __import__(mod_name)
        except Exception as e:  # noqa: BLE001
            failures.append((mod_name, type(e).__name__, str(e)))
    if failures:
        print("\nImport sanity check FAILED:")
        for name, kind, msg in failures:
            print(f"  - {name}: {kind}: {msg[:200]}")
        raise RuntimeError(
            "Um ou mais modulos quebrados. Causa comum no Colab: numpy/scipy/torch "
            "ficaram inconsistentes apos pip install. Solucao: "
            "Runtime > Restart runtime, depois execute esta celula novamente."
        )
    import numpy, scipy, pandas, sklearn, torch, transformers  # noqa: F401, E401
    print(f"\nVersoes: numpy={numpy.__version__}  scipy={scipy.__version__}  "
          f"pandas={pandas.__version__}  sklearn={sklearn.__version__}")
    print(f"          torch={torch.__version__}  transformers={transformers.__version__}")
    print("OK: economy_classifier importavel")


_sanity_check_imports()


## 2. Imports e configuracao

In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
from transformers import AutoModelForSequenceClassification

from economy_classifier.bert import (
    BertMulticlassConfig,
    BertTrainingConfig,
    MODEL_REGISTRY,
    predict_texts,
    train_bert_classifier,
    train_bert_multiclass,
)
from economy_classifier.datasets import MULTICLASS_TOP7, OTHER_LABEL, attach_multiclass_label
from economy_classifier.evaluation import (
    compute_binary_metrics,
    compute_confusion_matrix,
    compute_cost_metrics,
    compute_multiclass_metrics,
    compute_roc_auc,
    summarize_cv_metrics,
)
from economy_classifier.hyperparameter_search import (
    build_bert_search_space,
    random_search_bert,
)
from economy_classifier.project import (
    build_result_card,
    compute_artifact_size_mb,
    persist_result_card,
)

SEED = 42
MULTI_LABELS = list(MULTICLASS_TOP7) + [OTHER_LABEL]

# --- Hyperparameter search budget (per model x per task) ---
# A100: ~15-30 min per trial -> 25 trials = ~6-12h per (model, task) pair.
# Total search wall-clock for all 3 models x 2 tasks ~ 1-3 days. Reduce if needed.
N_ITER_BERT = 25
SEARCH_SPACE = build_bert_search_space()

# --- Which models to run. Comment out to skip ---
MODELS = [
    ("bertimbau",   MODEL_REGISTRY["bertimbau"]),
    ("finbert_ptbr", MODEL_REGISTRY["finbert_ptbr"]),
    ("deb3rta_base", MODEL_REGISTRY["deb3rta_base"]),
]
print("Modelos a treinar:", [m[0] for m in MODELS])
print(f"Search budget    : {N_ITER_BERT} trials x 2 tasks x {len(MODELS)} models "
      f"= {N_ITER_BERT * 2 * len(MODELS)} fits totais")

# --- Base BERT training overrides for the search (defaults that won't be sampled) ---
BASE_OVERRIDES = dict(
    max_length=256,
    per_device_eval_batch_size=64,
    early_stopping_patience=1,
    save_total_limit=1,
    gradient_checkpointing=False,
)


## 3. Carga dos splits e folds

In [ ]:
train = pd.read_parquet(SPLITS_DIR / "train.parquet")
val = pd.read_parquet(SPLITS_DIR / "val.parquet")
test = pd.read_parquet(SPLITS_DIR / "test.parquet")
cv_folds = json.loads((SPLITS_DIR / "cv_folds.json").read_text())

for split_name, split_df in [("train", train), ("val", val), ("test", test)]:
    if "label_multi" not in split_df.columns:
        if split_name == "train": train = attach_multiclass_label(train)
        elif split_name == "val": val = attach_multiclass_label(val)
        else: test = attach_multiclass_label(test)

train_val_pool = pd.concat([train, val]).sort_index()
print(f"train     : {len(train):>7,}  val: {len(val):>7,}  test: {len(test):>7,}")
print(f"pool 90%  : {len(train_val_pool):>7,}  cv_folds: {len(cv_folds)}")


## 4. Helpers compartilhados

In [ ]:
def model_size_mb(model_dir):
    return round(compute_artifact_size_mb(Path(model_dir)), 3)

def binary_metrics_from_preds(preds):
    m = compute_binary_metrics(preds["y_true"].values, preds["y_pred"].values)
    m["auc_roc"] = round(compute_roc_auc(preds["y_true"].values, preds["y_score"].values), 4)
    return m

def multiclass_metrics_from_preds(preds, labels):
    return compute_multiclass_metrics(preds["y_true"], preds["y_pred"], labels=labels)

def n_params_from_dir(model_dir):
    m = AutoModelForSequenceClassification.from_pretrained(model_dir)
    n = int(m.num_parameters())
    del m
    return n

def persist_search_log(search_result, search_dir):
    search_dir.mkdir(parents=True, exist_ok=True)
    out = search_dir / "search_result.json"
    out.write_text(json.dumps(search_result.to_dict(), indent=2, default=str))
    return out


## 5. Funcoes por regime

Cada `_eval_*` treina com a config vencedora da busca, persiste predicoes/metricas e emite o `result_card.json` com `hyperparameter_search`.

In [ ]:
def _eval_binary_fixed(model_id, model_key, config, search):
    run_dir = RUNS_BASE / f"{model_id}_binary_fixed_split"
    run_dir.mkdir(parents=True, exist_ok=True)
    result = train_bert_classifier(train, val, run_dir=run_dir, config=config)
    metrics = binary_metrics_from_preds(result["predictions"])
    n_params = n_params_from_dir(result["model_dir"])
    t0 = time.perf_counter()
    _ = predict_texts(val["text"].fillna("").tolist(), model_dir=result["model_dir"], method=model_key)
    inf_time = time.perf_counter() - t0
    cost = compute_cost_metrics(
        train_seconds=result["timing"]["train_seconds"], inference_seconds=inf_time,
        n_inference_samples=len(val), model_size_mb=model_size_mb(result["model_dir"]),
        n_parameters=n_params, hardware=HARDWARE,
    )
    result["predictions"].to_csv(run_dir / "predictions.csv", index=False)
    persist_result_card(build_result_card(
        model_id=model_id, task="binary", regime="fixed_split",
        metrics=metrics, cost=cost, config=config.to_dict(),
        n_train_samples=len(train), n_eval_samples=len(val),
        predictions_path=str(run_dir / "predictions.csv"),
        hyperparameter_search=search.card_payload(),
    ), run_dir)
    print(f"  [{model_id} binary fixed_split] f1={metrics['f1']:.4f}  auc={metrics['auc_roc']:.4f}")


def _eval_binary_cv(model_id, model_key, config, search):
    run_dir = RUNS_BASE / f"{model_id}_binary_cv_5fold"
    run_dir.mkdir(parents=True, exist_ok=True)
    fold_metrics, train_times, inf_times, all_preds = [], [], [], []
    n_params_last = size_mb_last = None
    for i, fold in enumerate(cv_folds):
        fold_dir = run_dir / f"fold_{i}"
        tr = train_val_pool.loc[fold["train_indices"]]
        va = train_val_pool.loc[fold["val_indices"]]
        res = train_bert_classifier(tr, va, run_dir=fold_dir, config=config)
        fm = binary_metrics_from_preds(res["predictions"])
        fold_metrics.append(fm)
        train_times.append(res["timing"]["train_seconds"])
        t0 = time.perf_counter()
        _ = predict_texts(va["text"].fillna("").tolist(), model_dir=res["model_dir"], method=model_key)
        inf_times.append(time.perf_counter() - t0)
        res["predictions"]["fold"] = i
        all_preds.append(res["predictions"])
        if i == len(cv_folds) - 1:
            n_params_last = n_params_from_dir(res["model_dir"])
            size_mb_last = model_size_mb(res["model_dir"])
        print(f"  [{model_id} binary cv fold {i}] f1={fm['f1']:.4f}  auc={fm['auc_roc']:.4f}")

    summary = summarize_cv_metrics(fold_metrics)
    cost = compute_cost_metrics(
        train_seconds=train_times, inference_seconds=inf_times,
        n_inference_samples=len(train_val_pool) // len(cv_folds),
        model_size_mb=size_mb_last, n_parameters=n_params_last, hardware=HARDWARE,
    )
    pd.concat(all_preds).to_csv(run_dir / "predictions.csv", index=False)
    persist_result_card(build_result_card(
        model_id=model_id, task="binary", regime="cv_5fold",
        metrics=summary, cost=cost, config=config.to_dict(),
        n_train_samples=len(train_val_pool), n_eval_samples=len(train_val_pool),
        predictions_path=str(run_dir / "predictions.csv"),
        notes="5-fold stratified CV on the 90% train+val pool (folds independent of search inner val).",
        hyperparameter_search=search.card_payload(),
    ), run_dir)
    print(f"  [{model_id} binary cv_5fold] f1 = {summary['f1_mean']:.4f} +/- {summary['f1_std']:.4f}")


def _eval_binary_test(model_id, model_key, config, search):
    run_dir = RUNS_BASE / f"{model_id}_binary_test_set"
    run_dir.mkdir(parents=True, exist_ok=True)
    result = train_bert_classifier(train_val_pool, test, run_dir=run_dir, config=config)
    metrics = binary_metrics_from_preds(result["predictions"])
    n_params = n_params_from_dir(result["model_dir"])
    t0 = time.perf_counter()
    _ = predict_texts(test["text"].fillna("").tolist(), model_dir=result["model_dir"], method=model_key)
    inf_time = time.perf_counter() - t0
    cost = compute_cost_metrics(
        train_seconds=result["timing"]["train_seconds"], inference_seconds=inf_time,
        n_inference_samples=len(test), model_size_mb=model_size_mb(result["model_dir"]),
        n_parameters=n_params, hardware=HARDWARE,
    )
    result["predictions"].to_csv(run_dir / "predictions.csv", index=False)
    persist_result_card(build_result_card(
        model_id=model_id, task="binary", regime="test_set",
        metrics=metrics, cost=cost, config=config.to_dict(),
        n_train_samples=len(train_val_pool), n_eval_samples=len(test),
        predictions_path=str(run_dir / "predictions.csv"),
        notes="Final single-shot evaluation on the held-out 10% test set, refit on train+val.",
        hyperparameter_search=search.card_payload(),
    ), run_dir)
    print(f"  [{model_id} binary test_set] f1={metrics['f1']:.4f}  auc={metrics['auc_roc']:.4f}")


def _eval_multi_fixed(model_id, model_key, config, search):
    run_dir = RUNS_BASE / f"{model_id}_multiclass_fixed_split"
    run_dir.mkdir(parents=True, exist_ok=True)
    result = train_bert_multiclass(train, val, label_column="label_multi", run_dir=run_dir, config=config)
    metrics = multiclass_metrics_from_preds(result["predictions"], labels=MULTI_LABELS)
    cost = compute_cost_metrics(
        train_seconds=result["timing"]["train_seconds"],
        inference_seconds=result["timing"]["inference_seconds"],
        n_inference_samples=len(val), model_size_mb=model_size_mb(result["model_dir"]),
        n_parameters=result["n_parameters"], hardware=HARDWARE,
    )
    result["predictions"].to_csv(run_dir / "predictions.csv", index=False)
    cm = compute_confusion_matrix(result["predictions"]["y_true"], result["predictions"]["y_pred"], labels=MULTI_LABELS)
    cm.to_csv(run_dir / "confusion_matrix.csv")
    persist_result_card(build_result_card(
        model_id=model_id, task="multiclass", regime="fixed_split",
        metrics=metrics, cost=cost, config=config.to_dict(),
        n_train_samples=len(train), n_eval_samples=len(val),
        predictions_path=str(run_dir / "predictions.csv"),
        hyperparameter_search=search.card_payload(),
    ), run_dir)
    print(f"  [{model_id} multi fixed_split] macro_f1={metrics['macro_f1']:.4f}  acc={metrics['accuracy']:.4f}")


def _eval_multi_cv(model_id, model_key, config, search):
    run_dir = RUNS_BASE / f"{model_id}_multiclass_cv_5fold"
    run_dir.mkdir(parents=True, exist_ok=True)
    fold_metrics, train_times, inf_times, all_preds = [], [], [], []
    n_params_last = size_mb_last = None
    for i, fold in enumerate(cv_folds):
        fold_dir = run_dir / f"fold_{i}"
        tr = train_val_pool.loc[fold["train_indices"]]
        va = train_val_pool.loc[fold["val_indices"]]
        res = train_bert_multiclass(tr, va, label_column="label_multi", run_dir=fold_dir, config=config)
        fm = multiclass_metrics_from_preds(res["predictions"], labels=MULTI_LABELS)
        fold_metrics.append(fm)
        train_times.append(res["timing"]["train_seconds"])
        inf_times.append(res["timing"]["inference_seconds"])
        res["predictions"]["fold"] = i
        all_preds.append(res["predictions"])
        if i == len(cv_folds) - 1:
            n_params_last = res["n_parameters"]
            size_mb_last = model_size_mb(res["model_dir"])
        print(f"  [{model_id} multi cv fold {i}] macro_f1={fm['macro_f1']:.4f}")

    summary = summarize_cv_metrics(fold_metrics)
    cost = compute_cost_metrics(
        train_seconds=train_times, inference_seconds=inf_times,
        n_inference_samples=len(train_val_pool) // len(cv_folds),
        model_size_mb=size_mb_last, n_parameters=n_params_last, hardware=HARDWARE,
    )
    pd.concat(all_preds).to_csv(run_dir / "predictions.csv", index=False)
    persist_result_card(build_result_card(
        model_id=model_id, task="multiclass", regime="cv_5fold",
        metrics=summary, cost=cost, config=config.to_dict(),
        n_train_samples=len(train_val_pool), n_eval_samples=len(train_val_pool),
        predictions_path=str(run_dir / "predictions.csv"),
        notes="5-fold stratified CV on the 90% train+val pool (folds independent of search inner val).",
        hyperparameter_search=search.card_payload(),
    ), run_dir)
    print(f"  [{model_id} multi cv_5fold] macro_f1 = {summary['macro_f1_mean']:.4f} +/- {summary['macro_f1_std']:.4f}")


def _eval_multi_test(model_id, model_key, config, search):
    run_dir = RUNS_BASE / f"{model_id}_multiclass_test_set"
    run_dir.mkdir(parents=True, exist_ok=True)
    result = train_bert_multiclass(train_val_pool, test, label_column="label_multi", run_dir=run_dir, config=config)
    metrics = multiclass_metrics_from_preds(result["predictions"], labels=MULTI_LABELS)
    cost = compute_cost_metrics(
        train_seconds=result["timing"]["train_seconds"],
        inference_seconds=result["timing"]["inference_seconds"],
        n_inference_samples=len(test), model_size_mb=model_size_mb(result["model_dir"]),
        n_parameters=result["n_parameters"], hardware=HARDWARE,
    )
    result["predictions"].to_csv(run_dir / "predictions.csv", index=False)
    cm = compute_confusion_matrix(result["predictions"]["y_true"], result["predictions"]["y_pred"], labels=MULTI_LABELS)
    cm.to_csv(run_dir / "confusion_matrix.csv")
    persist_result_card(build_result_card(
        model_id=model_id, task="multiclass", regime="test_set",
        metrics=metrics, cost=cost, config=config.to_dict(),
        n_train_samples=len(train_val_pool), n_eval_samples=len(test),
        predictions_path=str(run_dir / "predictions.csv"),
        notes="Final single-shot evaluation on the held-out 10% test set, refit on train+val.",
        hyperparameter_search=search.card_payload(),
    ), run_dir)
    print(f"  [{model_id} multi test_set] macro_f1={metrics['macro_f1']:.4f}")


## 6. Protocolo por modelo

`run_full_protocol(model_key, model_name)` executa: (1) busca binaria, (2) busca multiclasse, (3-5) 3 regimes binarios, (6-8) 3 regimes multiclasse. Emite 6 result_cards + 2 search logs.

In [ ]:
def run_full_protocol(model_key: str, model_name: str) -> dict:
    """Search + 6 regimes for one BERT model. Returns a summary dict.

    Emits to RUNS_BASE:
      - bert_{model_key}_search_binary/search_result.json
      - bert_{model_key}_search_multiclass/search_result.json
      - bert_{model_key}_{binary|multiclass}_{fixed_split|cv_5fold|test_set}/result_card.json
    """
    model_id = f"bert_{model_key}"
    print(f"\n{'='*72}\nMODEL: {model_id} ({model_name})\n{'='*72}\n")

    # ---- 1. SEARCH (binary) ----
    search_dir = RUNS_BASE / f"{model_id}_search_binary"
    print(f"[{model_id}] Hyperparameter search BINARY: {N_ITER_BERT} trials")
    search_bin = random_search_bert(
        train, val,
        model_name=model_name,
        work_dir=SEARCH_TRIALS_BASE / f"{model_id}_binary",
        label_column="label",
        multiclass=False,
        base_config_overrides=BASE_OVERRIDES,
        search_space=SEARCH_SPACE,
        n_iter=N_ITER_BERT,
        seed=SEED,
    )
    persist_search_log(search_bin, search_dir)
    print(f"[{model_id}] Best binary F1 (val) = {search_bin.best_score:.4f}  "
          f"({search_bin.search_seconds:.0f}s)")
    print(f"[{model_id}] Best params: {json.dumps(search_bin.best_params, indent=2, default=str)}")

    config_bin = BertTrainingConfig(
        model_name=model_name, seed=SEED,
        **{**BASE_OVERRIDES, **search_bin.best_params},
    )

    # ---- 2. SEARCH (multiclass) ----
    search_dir = RUNS_BASE / f"{model_id}_search_multiclass"
    print(f"\n[{model_id}] Hyperparameter search MULTICLASS: {N_ITER_BERT} trials")
    search_multi = random_search_bert(
        train, val,
        model_name=model_name,
        work_dir=SEARCH_TRIALS_BASE / f"{model_id}_multiclass",
        label_column="label_multi",
        multiclass=True,
        label_set=tuple(MULTI_LABELS),
        base_config_overrides=BASE_OVERRIDES,
        search_space=SEARCH_SPACE,
        n_iter=N_ITER_BERT,
        seed=SEED,
    )
    persist_search_log(search_multi, search_dir)
    print(f"[{model_id}] Best macro_F1 (val) = {search_multi.best_score:.4f}  "
          f"({search_multi.search_seconds:.0f}s)")
    print(f"[{model_id}] Best params: {json.dumps(search_multi.best_params, indent=2, default=str)}")

    config_multi = BertMulticlassConfig(
        model_name=model_name, seed=SEED, label_set=tuple(MULTI_LABELS),
        **{**BASE_OVERRIDES, **search_multi.best_params},
    )

    # ---- 3. BINARY: fixed_split / cv_5fold / test_set ----
    _eval_binary_fixed(model_id, model_key, config_bin, search_bin)
    _eval_binary_cv(model_id, model_key, config_bin, search_bin)
    _eval_binary_test(model_id, model_key, config_bin, search_bin)

    # ---- 4. MULTICLASS: fixed_split / cv_5fold / test_set ----
    _eval_multi_fixed(model_id, model_key, config_multi, search_multi)
    _eval_multi_cv(model_id, model_key, config_multi, search_multi)
    _eval_multi_test(model_id, model_key, config_multi, search_multi)

    return {
        "model_id": model_id,
        "binary_search_best": search_bin.best_score,
        "multiclass_search_best": search_multi.best_score,
        "binary_search_seconds": search_bin.search_seconds,
        "multiclass_search_seconds": search_multi.search_seconds,
    }


## 7. Loop sobre todos os modelos

**Atencao**: este loop pode levar 1-3 dias em A100 (3 modelos x (25 trials de busca + 5 folds de CV + 2 single shots) por tarefa). Considere rodar um modelo por vez ajustando `MODELS` na celula 5.

In [ ]:
all_summaries = []
for model_key, model_name in MODELS:
    summary = run_full_protocol(model_key, model_name)
    all_summaries.append(summary)
    # Free GPU memory between models
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n\n=== TODOS OS MODELOS COMPLETOS ===")
for s in all_summaries:
    print(json.dumps(s, indent=2, default=str))


## 8. Sumario — 18 result cards (3 modelos x 6 regimes)

In [ ]:
rows = []
for sub in sorted(RUNS_BASE.glob("bert_*")):
    card_path = sub / "result_card.json"
    if not card_path.exists():
        continue
    c = json.loads(card_path.read_text())
    if c.get("task") not in ("binary", "multiclass"):
        continue
    metrics = c["metrics"]
    primary = (
        metrics.get("f1") or metrics.get("f1_mean")
        or metrics.get("macro_f1") or metrics.get("macro_f1_mean")
    )
    rows.append({
        "model_id": c["model_id"],
        "task": c["task"], "regime": c["regime"],
        "primary": primary,
        "train_s": c["cost"].get("train_seconds_mean"),
        "inf_s":   c["cost"].get("inference_seconds_mean"),
        "size_mb": c["cost"].get("model_size_mb"),
        "params":  c["cost"].get("n_parameters"),
        "search_s": (c.get("hyperparameter_search") or {}).get("search_seconds"),
        "n_trials": (c.get("hyperparameter_search") or {}).get("n_trials"),
    })
summary_df = pd.DataFrame(rows).sort_values(["model_id", "task", "regime"]).reset_index(drop=True)
summary_df
